# 00. Environment and contract

Milestone 0 check that the project is wired up. Three things to see:

1. where the data folders are, and that none of them is inside the git repo;
2. which library versions this environment runs (pinned in `pyproject.toml`);
3. how Python builds, validates and writes the versioned JSON the app reads.

*This file is generated by `tools/build_notebooks.py`. Edit the spec there.*

## 1. Where data lives

Cboe, FRED and Yahoo data may only be used for personal, non-commercial
purposes, so it must never be committed to this public repo. `ivf.config`
points everything at one folder outside the repo, `$VOL_DATA_DIR` (default
`~/data/vol-projects`). The functions only compute paths; they never create
folders.

In [ ]:
import ivf
from ivf import config

print("ivf", ivf.__version__)
print("repo root:", config.repo_root())
layout = config.data_layout()
for name in ("root", "raw", "interim", "processed", "app_data_real"):
    path = getattr(layout, name)
    print(f"{name:14} {path}  (exists: {path.exists()})")

## 2. The numeric stack

ADR 0002 pins every direct dependency with `==` and locks the rest in
`uv.lock`, so this table is the same on every machine that runs `uv sync`.

Note the torch thread count. Torch uses every core by default, and on the
development machine that was measured 6.7 to 20 times *slower* than 8 threads
for a small LSTM. The LSTM code (milestone 4) therefore sets the count
explicitly with `torch.set_num_threads`.

In [ ]:
import sys

import arch
import numpy
import pandas
import pmdarima
import statsmodels
import torch

print("python", sys.version.split()[0])
for library in (numpy, pandas, statsmodels, arch, pmdarima, torch):
    print(f"{library.__name__:12} {library.__version__}")
print("torch.get_num_threads():", torch.get_num_threads(), "(the default)")

## 3. The JSON contract

Python computes, JSON carries, the app draws. Every file the app reads is an
*envelope* around a `payload`. The envelope says which contract version it
follows (`schema_version`), when and from which code it was made
(`generated_at`, `source_commit`, `source_dirty`), and whether the numbers are
`synthetic` or `real`.

Before the first commit exists, git has no commit to report. `source_commit`
is then 40 zeros and `source_dirty` is `true`: honest, and valid under the
schema.

In [ ]:
from ivf import contract

# None becomes JSON null. NaN would be refused (see section 4).
payload = {"values": [18.25, None, 21.5]}
envelope = contract.build_envelope("example", payload, "synthetic")

contract.validate_envelope(envelope)  # raises SchemaViolation if malformed
blob = contract.serialise(envelope)

print(blob.decode())
print("source_commit:", envelope["source_commit"], "| source_dirty:", envelope["source_dirty"])

## 4. What a violation looks like

The JSON Schemas in `schemas/app-data/` are the contract. A document that
breaks them is rejected with the location of every problem. Strict JSON also
has no NaN, so `serialise` refuses it instead of writing a file a browser
cannot parse.

In [ ]:
broken = dict(envelope, data_mode="paper")  # data_mode is a closed enum
try:
    contract.validate_envelope(broken)
except contract.SchemaViolation as error:
    print(error)

try:
    contract.serialise({"payload": {"x": float("nan")}})
except contract.ContractError as error:
    print(error)